# 02. Models and headline results

**What this notebook establishes.** Every number in the Results sections up to and
including the tolerance analysis, recomputed from the stored posteriors.

Nothing here is refitted. `2-model/02_canonical_fit.py`, `07_state_model.py` and
`09_aggregation_ladder.py` do the sampling and write the posteriors; this notebook reads
them. If a number below stops matching the manuscript, one of the two has changed.

**The single criterion.** A unit is *classified* when the 95% equal-tailed posterior
interval for its slope excludes zero. No other interval width is used anywhere in this
study, and the helper below is the only implementation of the rule.

In [1]:
import json
from pathlib import Path

import arviz as az
import numpy as np
import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
PROC, MODEL, RES = ROOT / "data/processed", ROOT / "2-model", ROOT / "3-results"

def check(label, computed, published, tol=1e-9):
    "Recompute a published value and fail loudly if the manuscript no longer matches."
    ok = abs(float(computed) - float(published)) <= tol
    print(f"{'OK  ' if ok else 'MISMATCH'}  {label}: manuscript={published}  recomputed={computed}")
    assert ok, f"{label}: manuscript says {published}, artefacts say {computed}"

def classified(draws, lo_q=0.025, hi_q=0.975):
    "The single criterion used everywhere in this study: 95% ETI excluding zero."
    lo, hi = np.quantile(draws, lo_q, axis=1), np.quantile(draws, hi_q, axis=1)
    return (lo > 0) | (hi < 0)

def slopes(idata, var="b"):
    return idata.posterior[var].stack(sample=("chain", "draw")).values

In [2]:
rate = az.from_netcdf(MODEL / "idata_rate.nc")
b = slopes(rate)
cls = classified(b)

mu_b = float(rate.posterior["mu_b"].mean())
print(f"national drift: {mu_b:+.4f} per year "
      f"= {100 * (np.exp(mu_b * 10) - 1):.1f}% per decade")
check("regions classified", cls.sum(), 86)
check("national drift, % per decade", round(100 * (np.exp(mu_b * 10) - 1), 1), -20.0)

national drift: -0.0223 per year = -20.0% per decade
OK    regions classified: manuscript=86  recomputed=86
OK    national drift, % per decade: manuscript=-20.0  recomputed=-20.0


## The aggregation ladder

The paper's central comparison. Four nested partitions of the same country and decade,
one specification, one criterion; only the unit changes.

The unpooled column matters because it contains no prior at all. It uses quasi-Poisson
with each unit's own dispersion, referred to **t on n-2 residual degrees of freedom**.
An earlier version of this analysis used the normal critical value of 1.96, which on 11
annual points runs at a 7-8% false positive rate rather than 5% and inflated every
unpooled figure by roughly half.

In [3]:
ladder = pd.read_csv(RES / "tables/aggregation_ladder.csv")
display(ladder[["level", "units", "median_deaths_per_unit",
                "classified", "pct_classified", "classified_unpooled", "pct_unpooled"]])

for lvl, hier, unp in [("municipality", 61, 384), ("immediate region", 81, 83),
                       ("state", 21, 19), ("macro-region", 0, 5)]:
    row = ladder[ladder.level == lvl].iloc[0]
    check(f"{lvl}: hierarchical", row.classified, hier)
    check(f"{lvl}: unpooled", row.classified_unpooled, unp)

,level,units,median_deaths_per_unit,classified,pct_classified,classified_unpooled,pct_unpooled
0,municipality,5570,7,61,1.1,384,7.4
1,immediate region,510,115,81,15.9,83,16.3
2,state,27,3052,21,77.8,19,70.4
3,macro-region,5,17106,0,0.0,5,100.0


OK    municipality: hierarchical: manuscript=61  recomputed=61
OK    municipality: unpooled: manuscript=384  recomputed=384
OK    immediate region: hierarchical: manuscript=81  recomputed=81
OK    immediate region: unpooled: manuscript=83  recomputed=83
OK    state: hierarchical: manuscript=21  recomputed=21
OK    state: unpooled: manuscript=19  recomputed=19
OK    macro-region: hierarchical: manuscript=0  recomputed=0
OK    macro-region: unpooled: manuscript=5  recomputed=5


### Why the macro-region rung is uninterpretable

With five units the model cannot separate the national mean from the unit deviations.
Each unit's marginal posterior inherits the hyperparameter uncertainty, so none resolves
— while unpooled, all five resolve with intervals far from zero. The rung is reported
for completeness, and as a symptom of the design at k = 5 rather than a statement about
macro-regions. Dropping it silently would be selective reporting.

In [4]:
macro = az.from_netcdf(MODEL / "idata_ladder_macro_region.nc")
bm = slopes(macro)
print(f"per-unit posterior SD : {bm.std(axis=1).round(4)}")
print(f"national drift SD     : {float(macro.posterior['mu_b'].std()):.4f}")
print("-> almost all of each unit's uncertainty is the national parameter's own.")

per-unit posterior SD : [0.0129 0.0127 0.0127 0.0125 0.0128]
national drift SD     : 0.0117
-> almost all of each unit's uncertainty is the national parameter's own.


## No unit is worsening, at any level

Reading point estimates at face value, 10 regions look like they are getting worse. None
survives its own interval. This is the contrast Figure 1 draws.

In [5]:
srate = pd.read_csv(MODEL / "slopes_rate.csv")
sstate = pd.read_csv(MODEL / "slopes_state.csv")
check("regions with a positive point estimate", (srate.b_mean > 0).sum(), 10)
check("regions classified AND positive", ((srate.b_mean > 0) & srate.classified_95).sum(), 0)
check("states classified", sstate.classified_95.sum(), 21)
check("states classified AND positive", ((sstate.b_mean > 0) & sstate.classified_95).sum(), 0)

OK    regions with a positive point estimate: manuscript=10  recomputed=10
OK    regions classified AND positive: manuscript=0  recomputed=0
OK    states classified: manuscript=21  recomputed=21
OK    states classified AND positive: manuscript=0  recomputed=0


## Departure from the national trajectory

Two different questions. *Did this unit change?* compares its slope to zero. *Is this
unit falling behind?* compares its slope to the national drift, and only the second
identifies a place to act on.

The second contrast is taken **against the shrinkage target**, so it is not an
independent test — the prior has already pulled every unit toward `mu_b`. That is why
heterogeneity is also tested outside the model, by Cochran's Q on unpooled state slopes
with each state's own dispersion, where neither shrinkage nor the prior can create or
conceal it. The two answers differ, and reporting only the first would be misleading.

In [6]:
vs_state = pd.read_csv(RES / "tables/vs_national_state.csv")
vs_reg   = pd.read_csv(RES / "tables/vs_national_region.csv")
check("states separable from the national drift", (vs_state.lagging | vs_state.leading).sum(), 0)
check("regions separable", (vs_reg.lagging | vs_reg.leading).sum(), 9)
check("regions ahead", vs_reg.leading.sum(), 8)
check("regions lagging", vs_reg.lagging.sum(), 1)

het = json.load(open(RES / "SUPPORTING_CHECKS.json"))["between_state_heterogeneity"]
print(f"\nOutside the model: Q={het['Q']} on {het['df']} df, p={het['p_value']}, "
      f"I2={het['I2_pct']}%, tau={het['tau']}/yr")
check("Cochran Q", het["Q"], 114.7, tol=0.05)

OK    states separable from the national drift: manuscript=0  recomputed=0
OK    regions separable: manuscript=9  recomputed=9
OK    regions ahead: manuscript=8  recomputed=8
OK    regions lagging: manuscript=1  recomputed=1

Outside the model: Q=114.7 on 26 df, p=4.17e-13, I2=77.3%, tau=0.0112/yr
OK    Cochran Q: manuscript=114.7  recomputed=114.7


## Design analysis

The classification rate depends on Brazil's own distribution of true slopes and on the
prior, so it is not a transportable property of the registration system. Simulating
known slopes at each region's real births and fitted dispersion gives one that is.

Two arms. Arm A gives every region the same true slope and refits unpooled: it measures
what a reader of one unit's series can detect. Arm B draws true slopes from the fitted
between-unit spread and refits the paper's own hierarchical model, which is the only
setting where that procedure can be evaluated, since a common true slope would drive the
between-unit variance to zero and make it trivially confident.

Read type S and type M alongside power. Unpooled, a declaration is not only rare but
unreliable: in the smallest regions the sign is wrong in about one declaration in nine and
the magnitude is exaggerated more than fivefold. Under the hierarchical procedure the sign
error falls below 2% everywhere and the magnitude is essentially unbiased.

In [7]:
two = pd.read_csv(RES / "tables/design_two_arm.csv")
display(two)

d = json.load(open(RES / "DESIGN_TWO_ARM.json"))
check("arm A, mean power at a national-sized change", d["arm_A_mean_power_at_1x"], 0.116)
check("arm A, declaration rate under a true zero", d["arm_A_null_declaration_rate"], 0.046, tol=0.0005)
check("arm B, mean power", d["arm_B_mean_power"], 0.195)
check("arm B, mean classification count", d["arm_B_classification_count"]["mean"], 99.1, tol=0.05)
print("\narm B classification range:", d["arm_B_classification_count"]["min"],
      "to", d["arm_B_classification_count"]["max"], "- contains the observed 86")

,arm,exposure_quintile,n,power,type_S,type_M,true_slope_x_national
0,"A: fixed 0.0x, unpooled",1,2080,0.048,0.0000,NaN,0.0
1,"A: fixed 0.0x, unpooled",2,2040,0.050,0.0000,NaN,0.0
2,"A: fixed 0.0x, unpooled",3,2080,0.045,0.0000,NaN,0.0
3,"A: fixed 0.0x, unpooled",4,1960,0.048,0.0000,NaN,0.0
4,"A: fixed 0.0x, unpooled",5,2040,0.040,0.0000,NaN,0.0
5,"A: fixed 0.5x, unpooled",1,2080,0.058,0.3000,10.01,0.5
6,"A: fixed 0.5x, unpooled",2,2040,0.053,0.2569,7.43,0.5
7,"A: fixed 0.5x, unpooled",3,2080,0.059,0.2295,6.32,0.5
8,"A: fixed 0.5x, unpooled",4,1960,0.073,0.1538,5.00,0.5
9,"A: fixed 0.5x, unpooled",5,2040,0.100,0.0739,3.23,0.5


OK    arm A, mean power at a national-sized change: manuscript=0.116  recomputed=0.116
OK    arm A, declaration rate under a true zero: manuscript=0.046  recomputed=0.0462
OK    arm B, mean power: manuscript=0.195  recomputed=0.195
OK    arm B, mean classification count: manuscript=99.1  recomputed=99.1

arm B classification range: 69 to 122 - contains the observed 86


## Certifying change is not the same as certifying stability

At a tolerance equal to the national drift, no region can be certified stable. That is
**arithmetic before it is empirical**: only one region in the country has a posterior
standard deviation small enough to be certified stable at that tolerance *even if its
true slope were exactly zero*. The manuscript states this precondition, because without
it the result reads as a discovery about Brazil rather than about precision.

In [8]:
rope = pd.read_csv(RES / "tables/rope_curve_rate.csv")
display(rope)

at1 = rope[rope.tolerance_x_national == 1.0].iloc[0]
check("changing at a tolerance of 1x", at1.drifting, 9)
check("stable at 1x", at1.credibly_flat, 0)
check("indeterminate at 1x (%)", round(100 * at1.indeterminate_share, 1), 98.2)

eligible = (srate.b_sd < abs(mu_b) / 1.96).sum()
check("regions precise enough to ever be certified stable at 1x", eligible, 1)

,tolerance_x_national,rope_halfwidth,drifting,credibly_flat,conclusive,indeterminate_share
0,0.25,0.005569,49,0,49,0.903922
1,0.50,0.011137,28,0,28,0.945098
2,1.00,0.022275,9,0,9,0.982353
3,2.00,0.044549,2,76,78,0.847059
4,3.00,0.066824,0,404,404,0.207843
5,4.00,0.089099,0,505,505,0.009804


OK    changing at a tolerance of 1x: manuscript=9  recomputed=9.0
OK    stable at 1x: manuscript=0  recomputed=0.0
OK    indeterminate at 1x (%): manuscript=98.2  recomputed=98.2
OK    regions precise enough to ever be certified stable at 1x: manuscript=1  recomputed=1
